---
toc: true
comments: true
layout: post
codemirror: true
title: RPN Calculator - Data Structures Lesson
description: Learn about Reverse Polish Notation, Stacks, ArrayLists, and HashMaps through building a calculator
courses: { csa: {week: 25} }
type: ccc
author: John Mortensen
permalink: /csa/rpn-lesson
---

## Introduction to Expression Evaluation

In mathematics, we write expressions like `3 + 4 * 5` using **infix notation** - the operator is *between* the operands. However, computers need to handle:
- **Operator precedence** (multiplication before addition)
- **Parentheses** for grouping
- **User input errors**

### What is Reverse Polish Notation (RPN)?

RPN (also called **postfix notation**) places operators *after* their operands:
- Infix: `3 + 4`
- RPN: `3 4 +`

**Why use RPN?**
- No need for parentheses
- No ambiguity about order of operations
- Easy to implement with a stack

### Key Data Structures We'll Use

1. **HashMap** - Fast lookup of operators and their properties
2. **ArrayList** - Store sequence of terms/tokens
3. **Stack** - Convert infix to RPN and calculate results

## Part 1: Token Class - Building Blocks

A `Token` represents the basic building block of a mathematical expression. It can be:
- An **operator** like `+`, `-`, `*`, `/`
- A **parenthesis** like `(` or `)`
- Later, we'll extend it to hold **values** too

### Key Concepts:
- **Precedence**: Which operators are calculated first (e.g., `*` before `+`)
- **BiFunction**: A functional interface that takes 2 inputs and returns 1 output
- **Telescoping Constructors**: Constructors that call other constructors

In [ ]:
// CODE_RUNNER: Token Class - Understanding operator precedence
import java.util.function.BiFunction;

public class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    // Constructor for passive token (used for parenthesis)
    public Token() {
        this('0');  // telescope to next constructor
    }

    // Constructor for simple token
    public Token(Character token) {
        this(token, 0, null, 0);
    }

    // Constructor for operators with precedence and calculation
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }

    // Getters
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }

    // Check if this token has higher precedence
    public boolean isPrecedent(Token token) {
        return this.precedence > token.getPrecedence();
    }

    // Perform calculation
    public Double calculate(Double operand1, Double operand2) {
        return this.calculation.apply(operand1, operand2);
    }

    public String toString() {
        return this.token.toString();
    }
}

// Test the Token class
Token plusToken = new Token('+', 4, (a, b) -> a + b, 2);
Token multiplyToken = new Token('*', 3, (a, b) -> a * b, 2);

System.out.println("Plus token: " + plusToken);
System.out.println("Multiply token: " + multiplyToken);
System.out.println("Plus precedence: " + plusToken.getPrecedence());
System.out.println("Multiply precedence: " + multiplyToken.getPrecedence());
System.out.println("Is multiply more precedent than plus? " + multiplyToken.isPrecedent(plusToken));
System.out.println("5 + 3 = " + plusToken.calculate(5.0, 3.0));
System.out.println("5 * 3 = " + multiplyToken.calculate(5.0, 3.0));

### Think About It:
- Why do we use `final` for instance variables?
- What does lower precedence number mean? (Hint: multiplication has precedence 3, addition has 4)
- How does `BiFunction` help us store the calculation logic?

## Part 2: TermOrOperator - Extending Tokens

The `TermOrOperator` class extends `Token` to handle:
- **Numeric values** (like `100`, `3.14`)
- **Operators** (like `+`, `-`, `*`, `/`)
- **Parentheses** (like `(`, `)`)

This uses **inheritance** - `TermOrOperator` IS-A `Token` with additional functionality.

In [ ]:
// CODE_RUNNER: TermOrOperator - Representing values and operators
import java.util.function.BiFunction;

public class TermOrOperator extends Token {
    private final String value;

    // Constructor for numeric values
    public TermOrOperator(String value) {
        this.value = value;
    }

    // Constructor for parenthesis
    public TermOrOperator(Character token) {
        super(token);
        this.value = null;
    }

    // Constructor for operators (2 arguments)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        super(token, precedence, calculation, 2);
        this.value = null;
    }

    // Constructor for operators (variable arguments, e.g., unary)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        super(token, precedence, calculation, numArgs);
        this.value = null;
    }

    public String getValue() {
        return value;
    }

    public String toString() {
        if (this.value == null) {
            return super.toString();
        }
        return this.value;
    }
}

// Test TermOrOperator
TermOrOperator num100 = new TermOrOperator("100");
TermOrOperator plusOp = new TermOrOperator('+', 4, (a, b) -> a + b);
TermOrOperator leftParen = new TermOrOperator('(');

System.out.println("Number: " + num100);
System.out.println("Operator: " + plusOp);
System.out.println("Parenthesis: " + leftParen);
System.out.println("Value of number: " + num100.getValue());
System.out.println("Value of operator: " + plusOp.getValue());

### Think About It:
- Why do we have multiple constructors?
- When is `value` null and when does it have a string?
- How does the `toString()` method handle both cases?

## Part 3: Tokens Class - Using HashMap for Fast Lookup

The `Tokens` class is a **collection** that stores operators and separators using a **HashMap**.

### Why HashMap?
- **O(1) lookup time** - instant access to operator by character
- **Key-Value pairs** - character (like '+') maps to TermOrOperator object
- Much faster than searching through an ArrayList

### Map Methods:
- `put(key, value)` - Add an entry
- `get(key)` - Retrieve value by key
- `containsKey(key)` - Check if key exists

In [ ]:
// CODE_RUNNER: Tokens - HashMap for operator storage
import java.util.Map;
import java.util.HashMap;
import java.util.function.BiFunction;

public class Tokens {
    private Map<Character, TermOrOperator> map;

    public Tokens() {
        this.map = new HashMap<>();
    }

    // Add parenthesis or separator
    public void put(Character token) {
        this.map.put(token, new TermOrOperator(token));
    }

    // Add operator with calculation (variable args)
    public void put(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.map.put(token, new TermOrOperator(token, precedence, calculation, numArgs));
    }

    // Add operator with calculation (2 args default)
    public void put(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        this.map.put(token, new TermOrOperator(token, precedence, calculation));
    }

    // Retrieve operator
    public TermOrOperator get(Character token) {
        return this.map.get(token);
    }

    // Get precedence
    public int getPrecedence(Character token) {
        return this.map.get(token).getPrecedence();
    }

    // Check if token exists
    public boolean contains(Character token) {
        return this.map.containsKey(token);
    }

    public String toString() {
        return this.map.toString();
    }
}

// Test Tokens collection
Tokens operators = new Tokens();
operators.put('+', 4, (a, b) -> a + b);
operators.put('-', 4, (a, b) -> a - b);
operators.put('*', 3, (a, b) -> a * b);
operators.put('/', 3, (a, b) -> a / b);
operators.put('^', 2, (a, b) -> Math.pow(a, b));

System.out.println("All operators: " + operators);
System.out.println("\nContains '+': " + operators.contains('+'));
System.out.println("Contains '&': " + operators.contains('&'));
System.out.println("\nPrecedence of '+': " + operators.getPrecedence('+'));
System.out.println("Precedence of '*': " + operators.getPrecedence('*'));
System.out.println("Precedence of '^': " + operators.getPrecedence('^'));
System.out.println("\n2 + 3 = " + operators.get('+').calculate(2.0, 3.0));
System.out.println("2 ^ 3 = " + operators.get('^').calculate(2.0, 3.0));

### Think About It:
- What's the Big-O complexity of HashMap lookup vs ArrayList search?
- Why do we have overloaded `put()` methods?
- What happens if we try to `get()` a token that doesn't exist?

## Part 4: Understanding the Algorithm

### Step 1: Tokenization
Split the expression string into individual parts:
- `"100 + 200 * 3"` → `["100", "+", "200", "*", "3"]`

### Step 2: Infix to RPN Conversion (Shunting Yard Algorithm)
Uses a **Stack** to reorder based on precedence:
- `"100 + 200 * 3"` → `["100", "200", "3", "*", "+"]`
- Read left to right, output values immediately
- Stack operators, pop when needed based on precedence

### Step 3: Evaluate RPN
Uses a **Stack** to calculate:
1. Push numbers onto stack
2. When operator appears, pop operands, calculate, push result
3. Final stack value is the answer

## Part 5: The Complete Calculator

Now we'll put it all together! The Calculator class:
1. **Tokenizes** the expression into parts
2. **Converts** from infix to RPN using a stack
3. **Evaluates** the RPN expression to get the result

### Three Key Methods:
- `termTokenizer()` - Parse string into ArrayList of tokens
- `termsToRPN()` - Convert infix to postfix using Stack
- `rpnToResult()` - Calculate final result using Stack

In [ ]:
// CODE_RUNNER: Complete Calculator - Infix to RPN to Result
import java.util.ArrayList;
import java.util.Stack;

public class Calculator {
    private final String expression;
    private ArrayList<TermOrOperator> terms = new ArrayList<>();
    private ArrayList<TermOrOperator> rpnTerms = new ArrayList<>();
    private Tokens operators = new Tokens();
    private Tokens seperators = new Tokens();
    private Double result = 0.0;

    public Calculator(String expression) {
        initOperators();
        initSeperators();
        this.expression = expression;
        this.termTokenizer();
        this.termsToRPN();
        this.rpnToResult();
    }

    private void initOperators() {
        operators.put('*', 3, (a, b) -> a * b);
        operators.put('/', 3, (a, b) -> a / b);
        operators.put('%', 3, (a, b) -> a % b);
        operators.put('+', 4, (a, b) -> a + b);
        operators.put('-', 4, (a, b) -> a - b);
        operators.put('^', 2, (a, b) -> Math.pow(a, b));
        operators.put('√', 1, (a, b) -> Math.sqrt(a), 1);
    }

    private void initSeperators() {
        seperators.put(' ');
        seperators.put('(');
        seperators.put(')');
    }

    private void termTokenizer() {
        int start = 0;
        StringBuilder multiCharTerm = new StringBuilder();
        for (int i = 0; i < this.expression.length(); i++) {
            Character c = this.expression.charAt(i);
            
            if (operators.contains(c) || seperators.contains(c)) {
                if (multiCharTerm.length() > 0) {
                    this.terms.add(new TermOrOperator(this.expression.substring(start, i)));
                }
                TermOrOperator t = operators.get(c);
                if (t == null) {
                    t = seperators.get(c);
                }
                if (t != null && t.getToken() != ' ') {
                    this.terms.add(t);
                }
                start = i + 1;
                multiCharTerm = new StringBuilder();
            } else {
                multiCharTerm.append(c);
            }
        }
        if (multiCharTerm.length() > 0) {
            this.terms.add(new TermOrOperator(this.expression.substring(start)));
        }
    }

    private void termsToRPN() {
        Stack<TermOrOperator> operatorStack = new Stack<>();

        for (TermOrOperator term : terms) {
            if (term.getToken() == '(') {
                operatorStack.push(term);
            } else if (term.getToken() == ')') {
                while (operatorStack.peek() != null && operatorStack.peek().getToken() != '(') {
                    rpnTerms.add(operatorStack.pop());
                }
                operatorStack.pop();
            } else if (operators.contains(term.getToken())) {
                while (!operatorStack.isEmpty() && operators.contains(operatorStack.peek().getToken()) && term.isPrecedent(operatorStack.peek())) {
                    rpnTerms.add(operatorStack.pop());
                }
                operatorStack.push(term);
            } else {
                this.rpnTerms.add(term);
            }
        }
        while (!operatorStack.isEmpty()) {
            rpnTerms.add(operatorStack.pop());
        }
    }

    private void rpnToResult() {
        Stack<Double> calcStack = new Stack<Double>();

        for (TermOrOperator term : this.rpnTerms) {
            if (operators.contains(term.getToken())) {
                Double operand1, operand2, result;
                if (term.getNumArgs() == 1) {
                    operand1 = calcStack.pop();
                    operand2 = 0.0;
                } else {
                    operand2 = calcStack.pop();
                    operand1 = calcStack.pop();
                }
                result = term.calculate(operand1, operand2);
                calcStack.push(result);
            } else {
                calcStack.push(Double.valueOf(term.getValue()));
            }
        }
        this.result = calcStack.pop();
    }

    public String toString() {
        return ("Original expression: " + this.expression + "\n" +
                "Tokenized expression: " + this.terms.toString() + "\n" +
                "Reverse Polish Notation: " + this.rpnTerms.toString() + "\n" +
                "Final result: " + String.format("%.2f", this.result));
    }

    public static void main(String[] args) {
        Calculator calc1 = new Calculator("100 + 200 * 3");
        System.out.println("Example 1: Simple Precedence\n" + calc1);
        System.out.println();

        Calculator calc2 = new Calculator("(100 + 200) * 3");
        System.out.println("Example 2: Parentheses\n" + calc2);
        System.out.println();

        Calculator calc3 = new Calculator("√(3^2 + 4^2)");
        System.out.println("Example 3: Pythagorean Theorem\n" + calc3);
    }
}

Calculator.main(null);

## Visual Walkthrough: How It Works

Let's trace through `"3 + 4 * 2"` step by step:

### Step 1: Tokenization
```
Input: "3 + 4 * 2"
Terms: [3, +, 4, *, 2]
```

### Step 2: Infix to RPN
```
Term | Stack      | RPN Output
-----|------------|------------
3    | []         | [3]
+    | [+]        | [3]
4    | [+]        | [3, 4]
*    | [+, *]     | [3, 4]        (* has higher precedence)
2    | [+, *]     | [3, 4, 2]
end  | []         | [3, 4, 2, *, +]  (pop all operators)
```

### Step 3: Evaluate RPN
```
Token | Stack      | Action
------|------------|--------
3     | [3]        | Push 3
4     | [3, 4]     | Push 4
2     | [3, 4, 2]  | Push 2
*     | [3, 8]     | Pop 2,4; Push 4*2=8
+     | [11]       | Pop 8,3; Push 3+8=11
```

**Final Result: 11** ✓

## Practice Exercise 1: Trace the Algorithm

Trace through the expression `"(2 + 3) * 4"` manually:

1. What is the tokenized expression?
2. What is the RPN expression?
3. What is the final result?
4. Now test it with code below!

In [ ]:
// CODE_RUNNER: Practice - Test your expression

// Test the expression (2 + 3) * 4
Calculator practice1 = new Calculator("(2 + 3) * 4");
System.out.println(practice1);

System.out.println("\n" + "=".repeat(50));

// Now try your own expression!
Calculator yourCalc = new Calculator("10 + 5 * 2 - 3");
System.out.println("\nYour Expression:\n" + yourCalc);

## Practice Exercise 2: Add a New Operator

Modify the calculator to support a new operator. Some ideas:
- **Factorial** `!` (unary operator)
- **Integer division** `//`
- **Minimum** `min`
- **Maximum** `max`

Think about:
- What precedence should it have?
- Is it unary (1 arg) or binary (2 args)?
- How do you implement the calculation?

In [ ]:
// CODE_RUNNER: Challenge - Add a floor division operator

// TODO: Modify Calculator to add floor division (//) operator
// Example: 7 // 2 should equal 3

// Hint: Add this line in initOperators():
// operators.put('f', 3, (a, b) -> Math.floor(a / b));
// Then replace "//" with "f" in your expression

Calculator challenge = new Calculator("7 / 2");
System.out.println("Regular division:\n" + challenge);

System.out.println("\nChallenge: Implement floor division to get 3 instead of 3.5");

## Key Takeaways

### Data Structures Used:
1. **HashMap** - O(1) lookup for operators
2. **ArrayList** - Store ordered sequence of tokens
3. **Stack** - LIFO for precedence and calculation

### Algorithms:
1. **Tokenization** - Parse string into meaningful parts
2. **Shunting Yard** - Convert infix to postfix (RPN)
3. **RPN Evaluation** - Stack-based calculation

### Design Patterns:
1. **Inheritance** - TermOrOperator extends Token
2. **Functional Programming** - BiFunction for calculations
3. **Encapsulation** - Each class has clear responsibility

### Real-World Applications:
- Compilers and interpreters
- Spreadsheet formula evaluation
- Scientific calculators
- Database query optimization

## Homework Challenges

### Basic (Required):
1. Test the calculator with 5 different expressions including parentheses
2. Manually trace one expression through all three stages
3. Explain why RPN doesn't need parentheses

### Intermediate:
4. Add support for a new binary operator (your choice)
5. Implement error handling for division by zero
6. Create a method to validate expressions before calculating

### Advanced:
7. Support multi-character operators (like "sqrt", "max", "min")
8. Add support for functions with multiple arguments
9. Implement a graphical user interface for the calculator
10. Extend to support variables (like "x = 5" then "x + 3")